In [3]:
import pandas as pd
import numpy as np
import re


# =========================
# 辅助函数
# =========================
def normalize_text(text):
    """统一清理文本里的空格、破折号等。"""
    if pd.isna(text):
        return ""
    s = str(text).strip().lower()
    s = s.replace("\xa0", " ")
    s = s.replace("–", "-").replace("—", "-")
    s = " ".join(s.split())
    return s


def collapse_spaced_thousands(text):
    """
    把带空格分组的大数字合并:
    - 5 000 -> 5000
    - 5 000 000 -> 5000000
    - $10 000 -> $10000
    只合并“每组正好 3 位”的千分位写法，避免乱改普通句子。
    """
    if not text:
        return text

    pattern = re.compile(r"(?<!\d)(\d{1,3}(?: \d{3})+)(?!\d)")

    prev = None
    s = text
    while prev != s:
        prev = s
        s = pattern.sub(lambda m: m.group(1).replace(" ", ""), s)

    return s


def contains_any(text, patterns):
    """只要 text 匹配任意一个 pattern，就返回 True。"""
    for pat in patterns:
        if re.search(pat, text):
            return True
    return False


def looks_like_uncertain_text(s):
    """看起来像不确定回答。"""
    uncertain_patterns = [
        r"\bnot sure\b",
        r"\bunsure\b",
        r"\bidk\b",
        r"\bi don't know\b",
        r"\bcannot decide\b",
        r"\bcan't decide\b",
        r"\bno idea\b",
    ]
    return contains_any(s, uncertain_patterns)


def looks_like_zero_text(s):
    """看起来像 0 金额回答。"""
    zero_patterns = [
        r"\b0+\b",
        r"\bzero\b",
        r"\bnothing\b",
        r"\bno money\b",
        r"\bfree\b",
    ]
    return contains_any(s, zero_patterns)


def scale_multiplier(scale):
    """
    把金额单位转成倍数。
    注意：这里故意不支持单独 'm'，避免 '$5000 max' 被错读成 '$5000 m'。
    """
    scale = (scale or "").lower().strip()

    if scale in {"k", "thousand"}:
        return 1_000
    elif scale in {"mil", "million"}:
        return 1_000_000
    elif scale in {"b", "billion"}:
        return 1_000_000_000
    else:
        return 1


def score_candidate(context):
    """
    给候选金额打分。
    分高 = 更像“真实愿付金额”
    分低 = 更像“假设金额 / 夸张金额”
    """
    score = 0

    realistic_patterns = [
        r"\bmax\b",
        r"\bat most\b",
        r"\bno more than\b",
        r"\bwould pay\b",
        r"\bi'd pay\b",
        r"\bwilling to pay\b",
        r"\bbecause i'm not\b",
        r"\bbecause i am not\b",
        r"\brealistically\b",
    ]

    hypothetical_patterns = [
        r"\bif i were\b",
        r"\bif i was\b",
        r"\bif i had\b",
        r"\bbillionaire\b",
        r"\bin a perfect world\b",
    ]

    if contains_any(context, realistic_patterns):
        score += 5

    if contains_any(context, hypothetical_patterns):
        score -= 5

    return score


def choose_best_payment_value(candidates, full_text):
    """
    有多个金额时，不直接 max(values)。
    规则：
    1. 先按上下文打分，优先“更像真实愿付金额”的候选。
    2. 如果同分，取最后出现的那个（很多回答是后半句修正前半句）。
    3. 如果整句明显是 depends / if i were 这种假设句，也优先最后一个候选。
    4. 最后才 fallback 到 max。
    """
    if not candidates:
        return np.nan

    scored = []
    for idx, item in enumerate(candidates):
        scored.append((score_candidate(item["context"]), idx, item["value"]))

    best_score = max(x[0] for x in scored)

    # 有明显“现实支付”信号时，优先这些候选
    if best_score > 0:
        best_group = [x for x in scored if x[0] == best_score]
        best_group.sort(key=lambda x: x[1])   # 按出现顺序
        return best_group[-1][2]              # 同分时取最后一个

    # 如果全文带有 depends / if i were，常见模式是“前面假设很大，后面现实很小”
    if re.search(r"\bdepends\b", full_text) or re.search(r"\bif i were\b", full_text):
        return candidates[-1]["value"]

    # 默认才取最大
    return max(item["value"] for item in candidates)


def parse_money_value(text):
    """
    尽量从自由文本中提取金额。
    改进点：
    1. 不再接受单独 'm'，避免 '$5000 max' -> '$5000 m'
    2. 支持:
       - $5000
       - 5000 dollars
       - 5k / 5 k
       - 5mil / 5 million
       - 2b / 2 billion
       - 80 000
    3. 多金额句子不再直接取最大值，而是按上下文选更合理的金额
    """
    if pd.isna(text):
        return np.nan

    s = normalize_text(text)

    if s in {"", "nan", "n/a", "na", "none"}:
        return np.nan

    s = collapse_spaced_thousands(s)

    # 先判断明显的 0
    if looks_like_zero_text(s):
        return 0.0

    num_pattern = r"(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?"

    # 这里故意去掉单独 m
    scale_pattern = r"(?:k|thousand|mil|million|b|billion)?"

    money_pattern = re.compile(
        rf"""
        (?:
            (?:cad\s*\$?|\$|usd\s*\$?)\s*
            (?P<num1>{num_pattern})
            \s*(?P<scale1>{scale_pattern})
        )
        |
        (?:
            (?P<num2>{num_pattern})
            \s*(?P<scale2>{scale_pattern})
            \s*(?:cad|usd|dollars?|bucks?)\b
        )
        |
        (?:
            (?P<num3>{num_pattern})
            \s+(?P<scale3>k|thousand|mil|million|b|billion)\b
        )
        |
        (?:
            (?P<num4>{num_pattern})(?P<scale4>k|mil|b)\b
        )
        |
        (?:
            \b(?P<num5>{num_pattern})\b
        )
        """,
        re.VERBOSE | re.IGNORECASE
    )

    candidates = []

    for match in money_pattern.finditer(s):
        num = None
        scale = ""

        if match.group("num1") is not None:
            num = match.group("num1")
            scale = match.group("scale1") or ""
        elif match.group("num2") is not None:
            num = match.group("num2")
            scale = match.group("scale2") or ""
        elif match.group("num3") is not None:
            num = match.group("num3")
            scale = match.group("scale3") or ""
        elif match.group("num4") is not None:
            num = match.group("num4")
            scale = match.group("scale4") or ""
        elif match.group("num5") is not None:
            num = match.group("num5")
            scale = ""

        if num is None:
            continue

        num_clean = num.replace(",", "")

        try:
            value = float(num_clean)
        except ValueError:
            continue

        value *= scale_multiplier(scale)

        # 记录上下文，用于后面判断哪个金额更合理
        start = max(0, match.start() - 35)
        end = min(len(s), match.end() + 35)
        context = s[start:end]

        candidates.append({
            "value": value,
            "context": context
        })

    if candidates:
        return choose_best_payment_value(candidates, s)

    # 没抓到数字，再看是不是“不确定回答”
    if looks_like_uncertain_text(s):
        return np.nan

    return np.nan


def parse_payment_column(series):
    """
    金额列清洗:
    1. 从文本中提取金额
    2. 负数改成 0
    3. 大于 324000000 的值直接 cap 到 324000000
    """
    x = series.apply(parse_money_value)

    negative_count = (x < 0).sum()
    x.loc[x < 0] = 0

    cap_value = 324_000_000
    capped_count = (x > cap_value).sum()
    x = x.clip(lower=0, upper=cap_value)

    return x, negative_count, cap_value, capped_count


def clean_count_column(series, upper_q=0.99):
    """
    count 列清洗:
    1. 转成数值
    2. 负数改成 0
    3. 用全局高分位数做上界
    4. clip 后 round，避免出现小数
    """
    x = pd.to_numeric(series, errors="coerce")

    negative_count = (x < 0).sum()
    x.loc[x < 0] = 0

    non_na = x.dropna()
    if len(non_na) == 0:
        return x, negative_count, None, 0

    upper = int(np.ceil(non_na.quantile(upper_q)))
    before = x.copy()
    x = x.clip(lower=0, upper=upper)
    x = x.round()

    capped_count = (before > x).sum()
    return x, negative_count, upper, capped_count


def clean_bounded_scale(series, lower, upper):
    """
    有固定范围的量表列:
    - 超出范围的值设为 NaN
    - 不做 quantile cap
    """
    x = pd.to_numeric(series, errors="coerce")
    invalid_count = ((x < lower) | (x > upper)).sum()
    x = x.where((x >= lower) & (x <= upper), np.nan)
    return x, invalid_count


# =========================
# 0. 读取数据
# =========================
df = pd.read_csv("data1.csv")

print("=" * 60)
print("Step 0: Read CSV")
print("=" * 60)
print("Original shape:", df.shape)


# =========================
# 1. 新增 label 列
# =========================
print("\n" + "=" * 60)
print("Step 1: Add label column")
print("=" * 60)

label_map = {
    "The Persistence of Memory": 0,
    "The Starry Night": 1,
    "The Water Lily Pond": 2
}

df["Painting"] = df["Painting"].astype(str).str.strip()
df["Painting"] = df["Painting"].replace("nan", np.nan)
df["label"] = df["Painting"].map(label_map)


# =========================
# 2. Likert 列转成 1~5
# =========================
print("\n" + "=" * 60)
print("Step 2: Convert Likert columns to 1~5")
print("=" * 60)

likert_cols = [
    "This art piece makes me feel sombre.",
    "This art piece makes me feel content.",
    "This art piece makes me feel calm.",
    "This art piece makes me feel uneasy."
]

likert_map = {
    "1 - Strongly disagree": 1,
    "2 - Disagree": 2,
    "3 - Neutral/Unsure": 3,
    "4 - Agree": 4,
    "5 - Strongly agree": 5
}

for col in likert_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace("nan", np.nan)
    df[col] = df[col].map(likert_map)


# =========================
# 3. 清洗金额列
# =========================
print("\n" + "=" * 60)
print("Step 3: Parse and cap payment column")
print("=" * 60)

pay_col = "How much (in Canadian dollars) would you be willing to pay for this painting?"

df[pay_col], neg_count_pay, pay_cap_value, pay_capped_count = parse_payment_column(df[pay_col])

print("Negative values replaced with 0:", neg_count_pay)
print("Cap value used:", pay_cap_value)
print("Values capped to 324000000:", pay_capped_count)
print("Max payment after cap:", df[pay_col].max())


# =========================
# 4. 定义数值列 / 文本列
# =========================
print("\n" + "=" * 60)
print("Step 4: Define numeric and text/categorical columns")
print("=" * 60)

intensity_col = "On a scale of 1–10, how intense is the emotion conveyed by the artwork?"

count_cols = [
    "How many prominent colours do you notice in this painting?",
    "How many objects caught your eye in the painting?"
]

numeric_cols = [
    intensity_col,
    count_cols[0],
    count_cols[1],
    "This art piece makes me feel sombre.",
    "This art piece makes me feel content.",
    "This art piece makes me feel calm.",
    "This art piece makes me feel uneasy."
]

text_or_categorical_cols = [
    "Painting",
    "Describe how this painting makes you feel.",
    "If you could purchase this painting, which room would you put that painting in?",
    "If you could view this art in person, who would you want to view it with?",
    "What season does this art piece remind you of?",
    "If this painting was a food, what would be?",
    "Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting."
]


# =========================
# 5. 按列处理异常值 / 非法值
# =========================
print("\n" + "=" * 60)
print("Step 5: Column-specific cleaning")
print("=" * 60)

for col in count_cols:
    df[col], negative_count, upper, capped_count = clean_count_column(df[col], upper_q=0.99)
    print(f"\nColumn: {col}")
    print("Negative values replaced with 0:", negative_count)
    print("Integer upper bound from global P99:", upper)
    print("Values clipped above upper bound:", capped_count)

print(f"\nColumn: {intensity_col}")
df[intensity_col], invalid_count = clean_bounded_scale(df[intensity_col], 1, 10)
print("Out-of-range values set to NaN:", invalid_count)

for col in likert_cols:
    df[col], invalid_count = clean_bounded_scale(df[col], 1, 5)
    print(f"\nColumn: {col}")
    print("Out-of-range values set to NaN:", invalid_count)


# =========================
# 6. 统计缺失值
# =========================
print("\n" + "=" * 60)
print("Step 6: Count missing values")
print("=" * 60)

missing_count = df.isna().sum(axis=1)


# =========================
# 7. 删除缺失 >= 4 的行
# =========================
print("\n" + "=" * 60)
print("Step 7: Drop rows with >= 4 missing values")
print("=" * 60)

df_clean = df[missing_count < 4].copy()
print("Shape after dropping:", df_clean.shape)


# =========================
# 8. payment 缺失值用全局 median 填补
# =========================
print("\n" + "=" * 60)
print("Step 8: Fill payment missing values with global median")
print("=" * 60)

payment_median = df_clean[pay_col].median()
payment_missing_before = df_clean[pay_col].isna().sum()

df_clean[pay_col] = df_clean[pay_col].fillna(payment_median)

payment_missing_after = df_clean[pay_col].isna().sum()

print("Payment global median:", payment_median)
print("Payment missing before fill:", payment_missing_before)
print("Payment missing after fill:", payment_missing_after)


# =========================
# 9. 其他数值列用全局中位数填补
# =========================
print("\n" + "=" * 60)
print("Step 9: Fill other numeric columns with global median")
print("=" * 60)

for col in numeric_cols:
    global_median = df_clean[col].median()
    missing_before = df_clean[col].isna().sum()
    df_clean[col] = df_clean[col].fillna(global_median)
    missing_after = df_clean[col].isna().sum()

    print(f"\nColumn: {col}")
    print("Global median:", global_median)
    print("Missing before fill:", missing_before)
    print("Missing after fill:", missing_after)

discrete_int_cols = [intensity_col] + count_cols + likert_cols

for col in discrete_int_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").round().astype("Int64")


# =========================
# 10. 文本/类别列用 Missing 填补
# =========================
print("\n" + "=" * 60)
print('Step 10: Fill text/categorical columns with "Missing"')
print("=" * 60)

for col in text_or_categorical_cols:
    missing_before = df_clean[col].isna().sum()
    df_clean[col] = df_clean[col].fillna("Missing")
    missing_after = df_clean[col].isna().sum()

    print(f"\nColumn: {col}")
    print("Missing before fill:", missing_before)
    print("Missing after fill:", missing_after)


# =========================
# 11. 最终检查
# =========================
print("\n" + "=" * 60)
print("Step 11: Final check")
print("=" * 60)

print("Final shape:", df_clean.shape)
print("\nRemaining missing values per column:")
print(df_clean.isna().sum())


# =========================
# 12. 保存
# =========================
output_file = "cleaned_data_final.csv"
df_clean.to_csv(output_file, index=False)

print("\n" + "=" * 60)
print("Step 12: Save cleaned file")
print("=" * 60)
print(f"Saved successfully to: {output_file}")

Step 0: Read CSV
Original shape: (1686, 16)

Step 1: Add label column

Step 2: Convert Likert columns to 1~5

Step 3: Parse and cap payment column
Negative values replaced with 0: 0
Cap value used: 324000000
Values capped to 324000000: 16
Max payment after cap: 324000000.0

Step 4: Define numeric and text/categorical columns

Step 5: Column-specific cleaning

Column: How many prominent colours do you notice in this painting?
Negative values replaced with 0: 0
Integer upper bound from global P99: 12
Values clipped above upper bound: 16

Column: How many objects caught your eye in the painting?
Negative values replaced with 0: 0
Integer upper bound from global P99: 15
Values clipped above upper bound: 14

Column: On a scale of 1–10, how intense is the emotion conveyed by the artwork?
Out-of-range values set to NaN: 1

Column: This art piece makes me feel sombre.
Out-of-range values set to NaN: 0

Column: This art piece makes me feel content.
Out-of-range values set to NaN: 0

Column: Thi

In [5]:
# =========================
# X. 把 4 个情绪态度列转成 1~5
# =========================
print("\n" + "=" * 60)
print("Step X: Convert Likert-scale text to numeric 1~5")
print("=" * 60)

likert_cols = [
    "This art piece makes me feel sombre.",
    "This art piece makes me feel content.",
    "This art piece makes me feel calm.",
    "This art piece makes me feel uneasy."
]

likert_map = {
    "Strongly disagree": 1,
    "Disagree": 2,
    "Neutral": 3,
    "Agree": 4,
    "Strongly agree": 5
}

for col in likert_cols:
    print(f"\nColumn: {col}")
    print("Unique values before mapping:")
    print(df[col].value_counts(dropna=False))

    # 去掉首尾空格后再映射
    df[col] = df[col].astype(str).str.strip()

    # 把原本字符串 nan 再变回真正缺失
    df[col] = df[col].replace("nan", np.nan)

    # 映射成数字
    df[col] = df[col].map(likert_map)

    print("Unique values after mapping:")
    print(df[col].value_counts(dropna=False))

print(df.head(12))


Step X: Convert Likert-scale text to numeric 1~5

Column: This art piece makes me feel sombre.
Unique values before mapping:
This art piece makes me feel sombre.
NaN    1686
Name: count, dtype: int64
Unique values after mapping:
This art piece makes me feel sombre.
NaN    1686
Name: count, dtype: int64

Column: This art piece makes me feel content.
Unique values before mapping:
This art piece makes me feel content.
NaN    1686
Name: count, dtype: int64
Unique values after mapping:
This art piece makes me feel content.
NaN    1686
Name: count, dtype: int64

Column: This art piece makes me feel calm.
Unique values before mapping:
This art piece makes me feel calm.
NaN    1686
Name: count, dtype: int64
Unique values after mapping:
This art piece makes me feel calm.
NaN    1686
Name: count, dtype: int64

Column: This art piece makes me feel uneasy.
Unique values before mapping:
This art piece makes me feel uneasy.
NaN    1686
Name: count, dtype: int64
Unique values after mapping:
This art

C:\Users\asus\AppData\Local\Temp\ipykernel_25796\609792523.py:32: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace("nan", np.nan)
C:\Users\asus\AppData\Local\Temp\ipykernel_25796\609792523.py:32: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace("nan", np.nan)
C:\Users\asus\AppData\Local\Temp\ipykernel_25796\609792523.py:32: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_ob

In [6]:
# import re
# import numpy as np
# import pandas as pd
#
# from sklearn.naive_bayes import BernoulliNB
# from sklearn.feature_extraction.text import CountVectorizer
#
#
# # =========================================================
# # 1. 基本配置
# # =========================================================
# CSV_PATH = "cleaned_data_final.csv"
# LABEL_COL = "label"
#
# FOOD_COL = "If this painting was a food, what would be?"
# FEEL_COL = "Describe how this painting makes you feel."
# SOUND_COL = "Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting."
#
# LABEL_TO_NAME = {
#     0: "The Persistence of Memory",
#     1: "The Starry Night",
#     2: "The Water Lily Pond"
# }
#
#
# # =========================================================
# # 2. 固定词表（按你提供的高频词）
# #    只做简单归一，所以保留原始词，再在 normalize 时做少量合并
# # =========================================================
# FOOD_WORDS = [
#     "salad", "cream", "ice", "soup", "blueberry", "cake", "bread", "pizza", "cheese",
#     "chocolate", "pie", "sandwich", "spaghetti", "apple", "pasta", "bowl", "cheesecake",
#     "tea", "fruit", "chicken", "steak", "strawberry", "rice", "noodles", "matcha",
#     "sweet", "blueberries", "watermelon", "icecream", "lemon", "beef", "smoothie",
#     "ramen", "stale", "water", "coffee", "sushi", "lettuce", "burger", "noodle",
#     "garden", "french", "toast", "sugar", "fish"
# ]
#
# FEEL_WORDS = [
#     "calm", "happy", "peaceful", "relaxed", "sad", "quiet", "nostalgic", "uneasy", "awe",
#     "serene", "wonder", "hopeful", "lonely", "anxious", "joyful", "excited",
#     "melancholic", "tranquil", "depressed", "unsettling"
# ]
#
# SOUND_WORDS = [
#     "slow", "piano", "calm", "soft", "quiet", "violin", "peaceful", "upbeat", "light",
#     "classical", "happy", "sad", "gentle", "birds", "tempo", "fast", "playing",
#     "chirping", "flute", "strings", "soothing", "guitar", "sombre", "relaxing", "warm",
#     "ambient", "slightly", "violins", "eerie", "cheerful", "emotional", "jazz",
#     "melancholic", "intense", "orchestral", "joyful", "steady", "orchestra", "noise",
#     "dark", "bass", "paced", "mellow", "life", "dreamy", "ticking", "serene", "harp",
#     "minor", "slowly"
# ]
#
#
# # =========================================================
# # 3. 只做轻量正规化
# # =========================================================
# ALIASES = {
#     "blueberries": "blueberry",
#     "noodles": "noodle",
#     "violins": "violin",
#     "slowly": "slow",
# }
#
#
# def normalize_text(text):
#     """lowercase + 去标点 + 少量复数归一"""
#     if pd.isna(text):
#         text = "missing"
#
#     text = str(text).lower()
#     text = text.replace("-", " ")
#     text = re.sub(r"[^a-z\s]", " ", text)
#
#     tokens = []
#     for tok in text.split():
#         tok = ALIASES.get(tok, tok)
#         tokens.append(tok)
#
#     return " ".join(tokens)
#
#
# def normalize_vocab(words):
#     """把词表里的重复/归一后重复项去掉"""
#     cleaned = []
#     seen = set()
#
#     for w in words:
#         w = ALIASES.get(w, w)
#         if w not in seen:
#             cleaned.append(w)
#             seen.add(w)
#
#     return cleaned
#
#
# # =========================================================
# # 4. 一个小包装：sklearn BernoulliNB + zero-hit fallback
# # =========================================================
# class KeywordBernoulliNB:
#     """
#     固定关键词词表的 BernoulliNB
#     - binary multi-hot
#     - predict_proba 时，如果一个词都没命中，就直接返回类先验 p(c)
#     """
#
#     def __init__(self, vocab, alpha=1.0):
#         self.vocab = normalize_vocab(vocab)
#         self.alpha = alpha
#
#         self.vectorizer = CountVectorizer(
#             vocabulary=self.vocab,
#             preprocessor=normalize_text,
#             tokenizer=str.split,
#             token_pattern=None,
#             binary=True,
#             lowercase=False
#         )
#
#         self.model = BernoulliNB(alpha=alpha)
#
#     def fit(self, texts, y):
#         texts = pd.Series(texts).fillna("missing")
#         X = self.vectorizer.fit_transform(texts)
#         self.model.fit(X, y)
#         return self
#
#     def _transform(self, texts):
#         if isinstance(texts, str):
#             texts = [texts]
#         texts = pd.Series(texts).fillna("missing")
#         return self.vectorizer.transform(texts)
#
#     def hit_count(self, texts):
#         X = self._transform(texts)
#         return np.asarray(X.sum(axis=1)).ravel()
#
#     def predict_proba(self, texts):
#         X = self._transform(texts)
#         probs = self.model.predict_proba(X)
#
#         # zero-hit: 完全没命中关键词 -> 直接退回类先验 p(c)
#         hits = np.asarray(X.sum(axis=1)).ravel()
#         zero_mask = (hits == 0)
#
#         if np.any(zero_mask):
#             prior = self.model.class_count_ / self.model.class_count_.sum()
#             probs[zero_mask] = prior
#
#         return probs
#
#     def predict(self, texts):
#         probs = self.predict_proba(texts)
#         return np.argmax(probs, axis=1)
#
#     def predict_one(self, text):
#         return int(self.predict([text])[0])
#
#     def predict_proba_one(self, text):
#         return self.predict_proba([text])[0]
#
#     def predict_name_one(self, text):
#         pred = self.predict_one(text)
#         return LABEL_TO_NAME[pred]
#
#
# # =========================================================
# # 5. 训练三个模型
# # =========================================================
# def train_text_nb_models(csv_path=CSV_PATH, alpha=1.0):
#     df = pd.read_csv(csv_path)
#
#     food_model = KeywordBernoulliNB(FOOD_WORDS, alpha=alpha).fit(df[FOOD_COL], df[LABEL_COL])
#     feeling_model = KeywordBernoulliNB(FEEL_WORDS, alpha=alpha).fit(df[FEEL_COL], df[LABEL_COL])
#     soundtrack_model = KeywordBernoulliNB(SOUND_WORDS, alpha=alpha).fit(df[SOUND_COL], df[LABEL_COL])
#
#     return food_model, feeling_model, soundtrack_model
#
#
# # =========================================================
# # 6. 你要的精简 predict / predict_proba 接口
# # =========================================================
# def predict_food(text, food_model):
#     return food_model.predict_one(text)
#
#
# def predict_proba_food(text, food_model):
#     return food_model.predict_proba_one(text)
#
#
# def predict_feeling(text, feeling_model):
#     return feeling_model.predict_one(text)
#
#
# def predict_proba_feeling(text, feeling_model):
#     return feeling_model.predict_proba_one(text)
#
#
# def predict_soundtrack(text, soundtrack_model):
#     return soundtrack_model.predict_one(text)
#
#
# def predict_proba_soundtrack(text, soundtrack_model):
#     return soundtrack_model.predict_proba_one(text)
#
#
# # =========================================================
# # 7. 可选：给后面的 random forest 生成文本特征
# #    这个很适合你后面做 85% train -> 15% val 的外层流程
# # =========================================================
# def build_nb_feature_frame(df, food_model, feeling_model, soundtrack_model):
#     food_proba = food_model.predict_proba(df[FOOD_COL].fillna("missing"))
#     feel_proba = feeling_model.predict_proba(df[FEEL_COL].fillna("missing"))
#     sound_proba = soundtrack_model.predict_proba(df[SOUND_COL].fillna("missing"))
#
#     out = pd.DataFrame({
#         "food_nb_p0": food_proba[:, 0],
#         "food_nb_p1": food_proba[:, 1],
#         "food_nb_p2": food_proba[:, 2],
#         "food_nb_pred": np.argmax(food_proba, axis=1),
#         "food_nb_hit_count": food_model.hit_count(df[FOOD_COL].fillna("missing")),
#         "food_nb_zero_hit": (food_model.hit_count(df[FOOD_COL].fillna("missing")) == 0).astype(int),
#
#         "feel_nb_p0": feel_proba[:, 0],
#         "feel_nb_p1": feel_proba[:, 1],
#         "feel_nb_p2": feel_proba[:, 2],
#         "feel_nb_pred": np.argmax(feel_proba, axis=1),
#         "feel_nb_hit_count": feeling_model.hit_count(df[FEEL_COL].fillna("missing")),
#         "feel_nb_zero_hit": (feeling_model.hit_count(df[FEEL_COL].fillna("missing")) == 0).astype(int),
#
#         "sound_nb_p0": sound_proba[:, 0],
#         "sound_nb_p1": sound_proba[:, 1],
#         "sound_nb_p2": sound_proba[:, 2],
#         "sound_nb_pred": np.argmax(sound_proba, axis=1),
#         "sound_nb_hit_count": soundtrack_model.hit_count(df[SOUND_COL].fillna("missing")),
#         "sound_nb_zero_hit": (soundtrack_model.hit_count(df[SOUND_COL].fillna("missing")) == 0).astype(int),
#     })
#
#     return out
#
#
# # =========================================================
# # 8. 示例
# # =========================================================
# if __name__ == "__main__":
#     food_model, feeling_model, soundtrack_model = train_text_nb_models("cleaned_data_final.csv", alpha=1.0)
#
#     food_text = "salad with fruit and tea, maybe matcha"
#     feel_text = "calm peaceful happy"
#     sound_text = "slow soft piano with a peaceful classical feeling"
#
#     print("food pred:", predict_food(food_text, food_model))
#     print("food proba:", predict_proba_food(food_text, food_model))
#     print("food name:", food_model.predict_name_one(food_text))
#     print()
#
#     print("feeling pred:", predict_feeling(feel_text, feeling_model))
#     print("feeling proba:", predict_proba_feeling(feel_text, feeling_model))
#     print("feeling name:", feeling_model.predict_name_one(feel_text))
#     print()
#
#     print("soundtrack pred:", predict_soundtrack(sound_text, soundtrack_model))
#     print("soundtrack proba:", predict_proba_soundtrack(sound_text, soundtrack_model))
#     print("soundtrack name:", soundtrack_model.predict_name_one(sound_text))
#
#
#     from sklearn.metrics import accuracy_score
#     y_true = df[LABEL_COL]
#
#     food_pred_all = food_model.predict(df[FOOD_COL])
#     feeling_pred_all = feeling_model.predict(df[FEEL_COL])
#     soundtrack_pred_all = soundtrack_model.predict(df[SOUND_COL])
#
#     food_acc = accuracy_score(y_true, food_pred_all)
#     feeling_acc = accuracy_score(y_true, feeling_pred_all)
#     soundtrack_acc = accuracy_score(y_true, soundtrack_pred_all)
#
#     print("food accuracy:", round(food_acc, 4))
#     print("feeling accuracy:", round(feeling_acc, 4))
#     print("soundtrack accuracy:", round(soundtrack_acc, 4))

In [10]:

# import re
# import numpy as np
# import pandas as pd
#
# from sklearn.naive_bayes import BernoulliNB
# from sklearn.feature_extraction.text import CountVectorizer
# from sklearn.ensemble import RandomForestClassifier
#
#
# # =========================================================
# # 1. fixed column names
# # =========================================================
# LABEL_COL = "label"
#
# INTENSITY_COL = "On a scale of 1–10, how intense is the emotion conveyed by the artwork?"
# FEEL_TEXT_COL = "Describe how this painting makes you feel."
# SOMBRE_COL = "This art piece makes me feel sombre."
# CONTENT_COL = "This art piece makes me feel content."
# CALM_COL = "This art piece makes me feel calm."
# UNEASY_COL = "This art piece makes me feel uneasy."
# COLOUR_COUNT_COL = "How many prominent colours do you notice in this painting?"
# OBJECT_COUNT_COL = "How many objects caught your eye in the painting?"
# PRICE_COL = "How much (in Canadian dollars) would you be willing to pay for this painting?"
# ROOM_COL = "If you could purchase this painting, which room would you put that painting in?"
# VIEW_WITH_COL = "If you could view this art in person, who would you want to view it with?"
# SEASON_COL = "What season does this art piece remind you of?"
# FOOD_COL = "If this painting was a food, what would be?"
# SOUND_COL = "Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting."
#
# NUMERIC_COLS = [
#     INTENSITY_COL,
#     COLOUR_COUNT_COL,
#     OBJECT_COUNT_COL,
#     PRICE_COL,
# ]
#
# LIKERT_COLS = [
#     SOMBRE_COL,
#     CONTENT_COL,
#     CALM_COL,
#     UNEASY_COL,
# ]
#
# FIXED_ROOM_TAGS = ["Bathroom", "Bedroom", "Office", "Living room", "Dining room"]
# FIXED_VIEW_WITH_TAGS = ["Friends", "Family members", "Coworkers/Classmates", "Strangers", "By yourself"]
# FIXED_SEASON_TAGS = ["Spring", "Summer", "Fall", "Winter"]
#
# LABEL_TO_NAME = {
#     0: "The Persistence of Memory",
#     1: "The Starry Night",
#     2: "The Water Lily Pond"
# }
#
#
# # =========================================================
# # 2. current NB vocab (temporary version)
# # =========================================================
# FOOD_WORDS = [
#     "salad", "cream", "ice", "soup", "blueberry", "cake", "bread", "pizza", "cheese",
#     "chocolate", "pie", "sandwich", "spaghetti", "apple", "pasta", "bowl", "cheesecake",
#     "tea", "fruit", "chicken", "steak", "strawberry", "rice", "noodles", "matcha",
#     "sweet", "blueberries", "watermelon", "icecream", "lemon", "beef", "smoothie",
#     "ramen", "stale", "water", "coffee", "sushi", "lettuce", "burger", "noodle",
#     "garden", "french", "toast", "sugar", "fish"
# ]
#
# FEEL_WORDS = [
#     "calm", "happy", "peaceful", "relaxed", "sad", "quiet", "nostalgic", "uneasy", "awe",
#     "serene", "wonder", "hopeful", "lonely", "anxious", "joyful", "excited",
#     "melancholic", "tranquil", "depressed", "unsettling"
# ]
#
# SOUND_WORDS = [
#     "slow", "piano", "calm", "soft", "quiet", "violin", "peaceful", "upbeat", "light",
#     "classical", "happy", "sad", "gentle", "birds", "tempo", "fast", "playing",
#     "chirping", "flute", "strings", "soothing", "guitar", "sombre", "relaxing", "warm",
#     "ambient", "slightly", "violins", "eerie", "cheerful", "emotional", "jazz",
#     "melancholic", "intense", "orchestral", "joyful", "steady", "orchestra", "noise",
#     "dark", "bass", "paced", "mellow", "life", "dreamy", "ticking", "serene", "harp",
#     "minor", "slowly"
# ]
#
#
# # =========================================================
# # 3. light normalization
# # =========================================================
# ALIASES = {
#     "blueberries": "blueberry",
#     "noodles": "noodle",
#     "violins": "violin",
#     "slowly": "slow",
# }
#
# def normalize_text(text):
#     if pd.isna(text):
#         text = "missing"
#     text = str(text).lower()
#     text = text.replace("-", " ")
#     text = re.sub(r"[^a-z\s]", " ", text)
#     tokens = [ALIASES.get(tok, tok) for tok in text.split()]
#     return " ".join(tokens)
#
# def normalize_vocab(words):
#     cleaned = []
#     seen = set()
#     for w in words:
#         w = ALIASES.get(w, w)
#         if w not in seen:
#             cleaned.append(w)
#             seen.add(w)
#     return cleaned
#
#
# # =========================================================
# # 4. keyword BernoulliNB
# # =========================================================
# class KeywordBernoulliNB:
#     def __init__(self, vocab, alpha=1.0):
#         self.vocab = normalize_vocab(vocab)
#         self.alpha = alpha
#         self.vectorizer = CountVectorizer(
#             vocabulary=self.vocab,
#             preprocessor=normalize_text,
#             tokenizer=str.split,
#             token_pattern=None,
#             binary=True,
#             lowercase=False,
#         )
#         self.model = BernoulliNB(alpha=alpha)
#
#     def fit(self, texts, y):
#         texts = pd.Series(texts).fillna("missing")
#         X = self.vectorizer.fit_transform(texts)
#         self.model.fit(X, y)
#         return self
#
#     def _transform(self, texts):
#         if isinstance(texts, str):
#             texts = [texts]
#         texts = pd.Series(texts).fillna("missing")
#         return self.vectorizer.transform(texts)
#
#     def predict_proba(self, texts):
#         X = self._transform(texts)
#         probs = self.model.predict_proba(X)
#         hits = np.asarray(X.sum(axis=1)).ravel()
#         zero_mask = (hits == 0)
#         if np.any(zero_mask):
#             prior = self.model.class_count_ / self.model.class_count_.sum()
#             probs[zero_mask] = prior
#         return probs
#
#     def predict(self, texts):
#         probs = self.predict_proba(texts)
#         return np.argmax(probs, axis=1)
#
#
# def train_text_nb_models(df, alpha=1.0):
#     food_model = KeywordBernoulliNB(FOOD_WORDS, alpha=alpha).fit(df[FOOD_COL], df[LABEL_COL])
#     feeling_model = KeywordBernoulliNB(FEEL_WORDS, alpha=alpha).fit(df[FEEL_TEXT_COL], df[LABEL_COL])
#     soundtrack_model = KeywordBernoulliNB(SOUND_WORDS, alpha=alpha).fit(df[SOUND_COL], df[LABEL_COL])
#     return food_model, feeling_model, soundtrack_model
#
#
# # =========================================================
# # 5. fixed multi-hot helpers
# # =========================================================
# def _split_multilabel(text):
#     if pd.isna(text):
#         return []
#     return [item.strip() for item in str(text).split(",") if item.strip()]
#
# def _build_fixed_multihot(df):
#     out = pd.DataFrame(index=df.index)
#
#     room_sets = df[ROOM_COL].apply(_split_multilabel)
#     who_sets = df[VIEW_WITH_COL].apply(_split_multilabel)
#     season_sets = df[SEASON_COL].apply(_split_multilabel)
#
#     for tag in FIXED_ROOM_TAGS:
#         out[f"room_{tag.replace(' ', '_').replace('/', '_')}"] = room_sets.apply(lambda items: int(tag in items))
#
#     for tag in FIXED_VIEW_WITH_TAGS:
#         out[f"view_with_{tag.replace(' ', '_').replace('/', '_')}"] = who_sets.apply(lambda items: int(tag in items))
#
#     for tag in FIXED_SEASON_TAGS:
#         out[f"season_{tag.replace(' ', '_').replace('/', '_')}"] = season_sets.apply(lambda items: int(tag in items))
#
#     return out
#
#
# # =========================================================
# # 6. NB probability features only (9 cols)
# # =========================================================
# def build_nb_probability_frame(df, food_model, feeling_model, soundtrack_model):
#     food_proba = food_model.predict_proba(df[FOOD_COL].fillna("missing"))
#     feel_proba = feeling_model.predict_proba(df[FEEL_TEXT_COL].fillna("missing"))
#     sound_proba = soundtrack_model.predict_proba(df[SOUND_COL].fillna("missing"))
#
#     return pd.DataFrame({
#         "food_nb_p0": food_proba[:, 0],
#         "food_nb_p1": food_proba[:, 1],
#         "food_nb_p2": food_proba[:, 2],
#         "feel_nb_p0": feel_proba[:, 0],
#         "feel_nb_p1": feel_proba[:, 1],
#         "feel_nb_p2": feel_proba[:, 2],
#         "sound_nb_p0": sound_proba[:, 0],
#         "sound_nb_p1": sound_proba[:, 1],
#         "sound_nb_p2": sound_proba[:, 2],
#     }, index=df.index)
#
#
# # =========================================================
# # 7. outer RF features
# # =========================================================
# def build_rf_features(df, food_model, feeling_model, soundtrack_model, include_label=True):
#     numeric_df = df[NUMERIC_COLS].apply(pd.to_numeric, errors="coerce").copy()
#     likert_df = df[LIKERT_COLS].apply(pd.to_numeric, errors="coerce").copy()
#     multihot_df = _build_fixed_multihot(df)
#     nb_df = build_nb_probability_frame(df, food_model, feeling_model, soundtrack_model)
#
#     X = pd.concat([numeric_df, likert_df, multihot_df, nb_df], axis=1)
#
#     if include_label:
#         y = df[LABEL_COL].astype(int).copy()
#         return X, y
#     return X
#
#
# # =========================================================
# # 8. train / predict RF
# # =========================================================
# def train_rf(df, alpha=1.0, n_estimators=300, max_depth=None, min_samples_leaf=1, random_state=42):
#     food_model, feeling_model, soundtrack_model = train_text_nb_models(df, alpha=alpha)
#     X, y = build_rf_features(df, food_model, feeling_model, soundtrack_model, include_label=True)
#
#     rf_model = RandomForestClassifier(
#         n_estimators=n_estimators,
#         max_depth=max_depth,
#         min_samples_leaf=min_samples_leaf,
#         random_state=random_state,
#     )
#     rf_model.fit(X, y)
#     return rf_model, food_model, feeling_model, soundtrack_model
#
#
# def predict_rf(df_new, rf_model, food_model, feeling_model, soundtrack_model):
#     X_new = build_rf_features(df_new, food_model, feeling_model, soundtrack_model, include_label=False)
#     pred = rf_model.predict(X_new)
#     proba = rf_model.predict_proba(X_new)
#     pred_name = [LABEL_TO_NAME[int(x)] for x in pred]
#
#     out = pd.DataFrame({
#         "pred": pred,
#         "pred_name": pred_name,
#         "p0": proba[:, 0],
#         "p1": proba[:, 1],
#         "p2": proba[:, 2],
#     }, index=df_new.index)
#
#     return out

In [18]:
import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score
#
# LABEL_COL = "label"
#
# def run_rf_train_val(csv_path, test_size=0.15, random_state=42):
#     # 1. 读入数据
#     df = pd.read_csv(csv_path)
#
#     # 2. 分层切分
#     train_df, val_df = train_test_split(
#         df,
#         test_size=test_size,
#         random_state=random_state,
#         stratify=df[LABEL_COL]
#     )
#
#     # 3. 只在 training split 上训练 RF + 3个NB子模型
#     rf_model, food_model, feeling_model, soundtrack_model = train_rf(
#         train_df,
#         alpha=1.0,
#         n_estimators=300,
#         max_depth=None,
#         min_samples_leaf=1,
#         random_state=random_state
#     )
#
#     # 4. training set 预测
#     train_pred_df = predict_rf(
#         train_df,
#         rf_model,
#         food_model,
#         feeling_model,
#         soundtrack_model
#     )
#     train_acc = accuracy_score(train_df[LABEL_COL], train_pred_df["pred"])
#
#     # 5. validation set 预测
#     val_pred_df = predict_rf(
#         val_df,
#         rf_model,
#         food_model,
#         feeling_model,
#         soundtrack_model
#     )
#     val_acc = accuracy_score(val_df[LABEL_COL], val_pred_df["pred"])
#
#     print("train size:", len(train_df))
#     print("validation size:", len(val_df))
#     print("training accuracy:", round(train_acc, 4))
#     print("validation accuracy:", round(val_acc, 4))
#
#     return {
#         "train_df": train_df,
#         "val_df": val_df,
#         "rf_model": rf_model,
#         "food_model": food_model,
#         "feeling_model": feeling_model,
#         "soundtrack_model": soundtrack_model,
#         "train_pred_df": train_pred_df,
#         "val_pred_df": val_pred_df,
#         "train_acc": train_acc,
#         "val_acc": val_acc,
#     }
#
# # 运行
# for i in range(1000):
#     result = run_rf_train_val("cleaned_data_final.csv", test_size=0.25, random_state=i)

train size: 1207
validation size: 403
training accuracy: 1.0
validation accuracy: 0.8933
train size: 1207
validation size: 403
training accuracy: 1.0
validation accuracy: 0.8834
train size: 1207
validation size: 403
training accuracy: 1.0
validation accuracy: 0.9057
train size: 1207
validation size: 403
training accuracy: 1.0
validation accuracy: 0.9156
train size: 1207
validation size: 403
training accuracy: 1.0
validation accuracy: 0.9007
train size: 1207
validation size: 403
training accuracy: 1.0
validation accuracy: 0.8859
train size: 1207
validation size: 403
training accuracy: 1.0
validation accuracy: 0.9032
train size: 1207
validation size: 403
training accuracy: 1.0
validation accuracy: 0.9032
train size: 1207
validation size: 403
training accuracy: 1.0
validation accuracy: 0.9007
train size: 1207
validation size: 403
training accuracy: 1.0
validation accuracy: 0.9057
train size: 1207
validation size: 403
training accuracy: 1.0
validation accuracy: 0.8933
train size: 1207
vali

KeyboardInterrupt: 

In [6]:
# =========================
# Code Block 1
# 超参都放在最上面，方便调参
# =========================

import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.naive_bayes import BernoulliNB


# -------------------------
# Hyperparameters / Config
# -------------------------
CSV_PATH = "cleaned_data_final.csv"

LABEL_COL = "label"
FOOD_COL = "If this painting was a food, what would be?"
FEEL_COL = "Describe how this painting makes you feel."
SOUND_COL = "Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting."

FOOD_K = 40
FEELING_K = 38
SOUNDTRACK_K = 65

NB_ALPHA = 1.0
MIN_TOKEN_LEN = 2

EXCLUDE_MISSING_FROM_VOCAB = True
ZERO_HIT_RETURNS_PRIOR = True
SORT_TIES_ALPHABETICALLY = True

LABEL_TO_NAME = {
    0: "The Persistence of Memory",
    1: "The Starry Night",
    2: "The Water Lily Pond"
}

# 少量人工归一，仍然属于“简单复数/变体归一”
ALIASES = {
    "blueberries": "blueberry",
    "noodles": "noodle",
    "violins": "violin",
    "slowly": "slow",
    "icecream": "ice",
}


stop_words = {
    "'d",
    "'ll",
    "'m",
    "'re",
    "'s",
    "'ve",
    "a",
    "about",
    "above",
    "across",
    "after",
    "afterwards",
    "again",
    "against",
    "all",
    "almost",
    "alone",
    "along",
    "already",
    "also",
    "although",
    "always",
    "am",
    "among",
    "amongst",
    "amoungst",
    "amount",
    "an",
    "and",
    "another",
    "any",
    "anyhow",
    "anyone",
    "anything",
    "anyway",
    "anywhere",
    "are",
    "around",
    "as",
    "at",
    "back",
    "be",
    "became",
    "because",
    "become",
    "becomes",
    "becoming",
    "been",
    "before",
    "beforehand",
    "behind",
    "being",
    "below",
    "beside",
    "besides",
    "between",
    "beyond",
    "bill",
    "both",
    "bottom",
    "but",
    "by",
    "ca",
    "call",
    "can",
    "cannot",
    "cant",
    "co",
    "con",
    "could",
    "couldnt",
    "cry",
    "de",
    "describe",
    "detail",
    "did",
    "do",
    "does",
    "doing",
    "done",
    "down",
    "due",
    "during",
    "each",
    "eg",
    "eight",
    "either",
    "eleven",
    "else",
    "elsewhere",
    "empty",
    "enough",
    "etc",
    "even",
    "ever",
    "every",
    "everyone",
    "everything",
    "everywhere",
    "except",
    "few",
    "fifteen",
    "fifty",
    "fill",
    "find",
    "fire",
    "first",
    "five",
    "for",
    "former",
    "formerly",
    "forty",
    "found",
    "four",
    "from",
    "front",
    "full",
    "further",
    "get",
    "give",
    "go",
    "had",
    "has",
    "hasnt",
    "have",
    "he",
    "hence",
    "her",
    "here",
    "hereafter",
    "hereby",
    "herein",
    "hereupon",
    "hers",
    "herself",
    "him",
    "himself",
    "his",
    "how",
    "however",
    "hundred",
    "i",
    "ie",
    "if",
    "in",
    "inc",
    "indeed",
    "interest",
    "into",
    "is",
    "it",
    "its",
    "itself",
    "just",
    "keep",
    "last",
    "latter",
    "latterly",
    "least",
    "less",
    "ltd",
    "made",
    "make",
    "many",
    "may",
    "me",
    "meanwhile",
    "might",
    "mill",
    "mine",
    "more",
    "moreover",
    "most",
    "mostly",
    "move",
    "much",
    "must",
    "my",
    "myself",
    "n't",
    "name",
    "namely",
    "neither",
    "never",
    "nevertheless",
    "next",
    "nine",
    "no",
    "nobody",
    "none",
    "noone",
    "nor",
    "not",
    "nothing",
    "now",
    "nowhere",
    "n‘t",
    "n’t",
    "of",
    "off",
    "often",
    "on",
    "once",
    "one",
    "only",
    "onto",
    "or",
    "other",
    "others",
    "otherwise",
    "our",
    "ours",
    "ourselves",
    "out",
    "over",
    "own",
    "part",
    "per",
    "perhaps",
    "please",
    "put",
    "quite",
    "rather",
    "re",
    "really",
    "regarding",
    "same",
    "say",
    "see",
    "seem",
    "seemed",
    "seeming",
    "seems",
    "serious",
    "several",
    "she",
    "should",
    "show",
    "side",
    "since",
    "sincere",
    "six",
    "sixty",
    "so",
    "some",
    "somehow",
    "someone",
    "something",
    "sometime",
    "sometimes",
    "somewhere",
    "still",
    "such",
    "system",
    "take",
    "ten",
    "than",
    "that",
    "the",
    "their",
    "them",
    "themselves",
    "then",
    "thence",
    "there",
    "thereafter",
    "thereby",
    "therefore",
    "therein",
    "thereupon",
    "these",
    "they",
    "thick",
    "thin",
    "third",
    "this",
    "those",
    "though",
    "three",
    "through",
    "throughout",
    "thru",
    "thus",
    "to",
    "together",
    "too",
    "top",
    "toward",
    "towards",
    "twelve",
    "twenty",
    "two",
    "un",
    "under",
    "unless",
    "until",
    "up",
    "upon",
    "us",
    "used",
    "using",
    "various",
    "very",
    "via",
    "was",
    "we",
    "well",
    "were",
    "what",
    "whatever",
    "when",
    "whence",
    "whenever",
    "where",
    "whereafter",
    "whereas",
    "whereby",
    "wherein",
    "whereupon",
    "wherever",
    "whether",
    "which",
    "while",
    "whither",
    "who",
    "whoever",
    "whole",
    "whom",
    "whose",
    "why",
    "will",
    "with",
    "within",
    "without",
    "would",
    "yet",
    "you",
    "your",
    "yours",
    "yourself",
    "yourselves",
    "‘d",
    "‘ll",
    "‘m",
    "‘re",
    "‘s",
    "‘ve",
    "’d",
    "’ll",
    "’m",
    "’re",
    "’s",
    "’ve"
}


def simple_singularize(token: str) -> str:
    """只做很轻量的单复数归一。"""
    if token in ALIASES:
        return ALIASES[token]

    if len(token) <= 3:
        return token

    if token.endswith("ies") and len(token) > 4:
        return token[:-3] + "y"

    if token.endswith("es") and len(token) > 4 and not token.endswith("ses"):
        return token[:-2]

    if token.endswith("s") and len(token) > 3 and not token.endswith("ss"):
        return token[:-1]

    return token


def tokenize_text(text) -> list[str]:
    """
    只按你要求做：
    1) lowercase
    2) 去标点
    3) 简单复数归一
    4) 去 stopwords
    """
    if pd.isna(text):
        text = "missing"

    text = str(text).lower()
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("-", " ")
    text = re.sub(r"[^a-z'\s]", " ", text)

    tokens = []
    for raw_tok in text.split():
        tok = raw_tok.strip("'")
        if not tok:
            continue

        tok = simple_singularize(tok)

        if len(tok) < MIN_TOKEN_LEN:
            continue
        if tok in stop_words:
            continue

        tokens.append(tok)

    return tokens


def select_top_k_vocab_from_training(texts, k: int) -> tuple[list[str], dict]:
    """
    只用传入的 training texts 选 top-k。
    频率定义：document frequency
    """
    if k <= 0:
        raise ValueError("k must be > 0.")

    doc_counter = Counter()

    for text in pd.Series(texts).fillna("missing"):
        uniq_tokens = set(tokenize_text(text))

        if EXCLUDE_MISSING_FROM_VOCAB:
            uniq_tokens.discard("missing")

        for tok in uniq_tokens:
            doc_counter[tok] += 1

    if not doc_counter:
        raise ValueError("No usable tokens found in the training data after preprocessing.")

    items = list(doc_counter.items())

    if SORT_TIES_ALPHABETICALLY:
        items.sort(key=lambda x: (-x[1], x[0]))
    else:
        items.sort(key=lambda x: -x[1])

    vocab = [tok for tok, _ in items[:k]]
    return vocab, dict(doc_counter)


class TopKBernoulliNB:
    """
    训练时：
    - 从 training set 里选 top-k vocabulary
    - 用 binary multi-hot 向量训练 BernoulliNB

    预测时：
    - predict_proba 返回 [p(c=0), p(c=1), p(c=2)]
    - 如果 zero-hit，则直接回退到类先验 p(c)
    """

    def __init__(self, k: int, alpha: float = NB_ALPHA):
        self.k = k
        self.alpha = alpha

        self.vocab_ = None
        self.vocab_index_ = None
        self.doc_freq_ = None
        self.model_ = None

    def fit(self, texts, y):
        self.vocab_, self.doc_freq_ = select_top_k_vocab_from_training(texts, self.k)
        self.vocab_index_ = {tok: i for i, tok in enumerate(self.vocab_)}

        X = self._vectorize(texts)

        if X.shape[1] == 0:
            raise ValueError("Vocabulary is empty after top-k selection.")

        self.model_ = BernoulliNB(alpha=self.alpha)
        self.model_.fit(X, np.asarray(y, dtype=int))
        return self

    def _vectorize_one(self, text) -> np.ndarray:
        row = np.zeros(len(self.vocab_), dtype=np.int8)

        uniq_tokens = set(tokenize_text(text))
        for tok in uniq_tokens:
            j = self.vocab_index_.get(tok)
            if j is not None:
                row[j] = 1

        return row

    def _vectorize(self, texts) -> np.ndarray:
        if isinstance(texts, str):
            texts = [texts]

        texts = pd.Series(texts).fillna("missing")
        X = np.zeros((len(texts), len(self.vocab_)), dtype=np.int8)

        for i, text in enumerate(texts):
            X[i] = self._vectorize_one(text)

        return X

    def hit_count(self, texts) -> np.ndarray:
        X = self._vectorize(texts)
        return X.sum(axis=1)

    def predict_proba(self, texts) -> np.ndarray:
        X = self._vectorize(texts)
        probs = self.model_.predict_proba(X)

        if ZERO_HIT_RETURNS_PRIOR:
            hits = X.sum(axis=1)
            zero_mask = hits == 0
            if np.any(zero_mask):
                prior = self.model_.class_count_ / self.model_.class_count_.sum()
                probs[zero_mask] = prior

        return probs

    def predict(self, texts) -> np.ndarray:
        probs = self.predict_proba(texts)
        return np.argmax(probs, axis=1)

    def predict_proba_one(self, text) -> np.ndarray:
        return self.predict_proba([text])[0]

    def predict_one(self, text) -> int:
        return int(self.predict([text])[0])

    def predict_name_one(self, text) -> str:
        pred = self.predict_one(text)
        return LABEL_TO_NAME[pred]


from sklearn.metrics import accuracy_score

# =========================
# 训练三个文本模型
# =========================
food_model = TopKBernoulliNB(k=FOOD_K, alpha=NB_ALPHA)
feeling_model = TopKBernoulliNB(k=FEELING_K, alpha=NB_ALPHA)
soundtrack_model = TopKBernoulliNB(k=SOUNDTRACK_K, alpha=NB_ALPHA)

food_model.fit(df[FOOD_COL], df[LABEL_COL])
feeling_model.fit(df[FEEL_COL], df[LABEL_COL])
soundtrack_model.fit(df[SOUND_COL], df[LABEL_COL])

# =========================
# 在当前 df 上测试 accuracy
# =========================
y_true = df[LABEL_COL].to_numpy()

food_pred = food_model.predict(df[FOOD_COL])
feeling_pred = feeling_model.predict(df[FEEL_COL])
soundtrack_pred = soundtrack_model.predict(df[SOUND_COL])

food_acc = accuracy_score(y_true, food_pred)
feeling_acc = accuracy_score(y_true, feeling_pred)
soundtrack_acc = accuracy_score(y_true, soundtrack_pred)

print("food accuracy:", round(food_acc, 4))
print("feeling accuracy:", round(feeling_acc, 4))
print("soundtrack accuracy:", round(soundtrack_acc, 4))

food accuracy: 0.6109
feeling accuracy: 0.6501
soundtrack accuracy: 0.5759


In [8]:
# =========================
# Code Block 2
# 第二段直接接第一段后面
# =========================

# -------------------------
# 读数据
# -------------------------
def load_cleaned_data(csv_path: str = CSV_PATH) -> pd.DataFrame:
    return pd.read_csv(csv_path)


# -------------------------
# 训练三个 NB
# 这里一定是传入 df_train
# 也就是你外层 split 后的 training set
# -------------------------
def train_food_nb(df_train: pd.DataFrame, k: int = FOOD_K, alpha: float = NB_ALPHA) -> TopKBernoulliNB:
    model = TopKBernoulliNB(k=k, alpha=alpha)
    model.fit(df_train[FOOD_COL], df_train[LABEL_COL])
    return model


def train_feeling_nb(df_train: pd.DataFrame, k: int = FEELING_K, alpha: float = NB_ALPHA) -> TopKBernoulliNB:
    model = TopKBernoulliNB(k=k, alpha=alpha)
    model.fit(df_train[FEEL_COL], df_train[LABEL_COL])
    return model


def train_soundtrack_nb(df_train: pd.DataFrame, k: int = SOUNDTRACK_K, alpha: float = NB_ALPHA) -> TopKBernoulliNB:
    model = TopKBernoulliNB(k=k, alpha=alpha)
    model.fit(df_train[SOUND_COL], df_train[LABEL_COL])
    return model


def train_text_nb_models(
    df_train: pd.DataFrame,
    food_k: int = FOOD_K,
    feeling_k: int = FEELING_K,
    soundtrack_k: int = SOUNDTRACK_K,
    alpha: float = NB_ALPHA
):
    food_model = train_food_nb(df_train, k=food_k, alpha=alpha)
    feeling_model = train_feeling_nb(df_train, k=feeling_k, alpha=alpha)
    soundtrack_model = train_soundtrack_nb(df_train, k=soundtrack_k, alpha=alpha)
    return food_model, feeling_model, soundtrack_model


# -------------------------
# 你要的精简 predict / predict_proba 接口
# -------------------------
def predict_food(text: str, food_model: TopKBernoulliNB) -> int:
    return food_model.predict_one(text)


def predict_proba_food(text: str, food_model: TopKBernoulliNB) -> np.ndarray:
    return food_model.predict_proba_one(text)


def predict_feeling(text: str, feeling_model: TopKBernoulliNB) -> int:
    return feeling_model.predict_one(text)


def predict_proba_feeling(text: str, feeling_model: TopKBernoulliNB) -> np.ndarray:
    return feeling_model.predict_proba_one(text)


def predict_soundtrack(text: str, soundtrack_model: TopKBernoulliNB) -> int:
    return soundtrack_model.predict_one(text)


def predict_proba_soundtrack(text: str, soundtrack_model: TopKBernoulliNB) -> np.ndarray:
    return soundtrack_model.predict_proba_one(text)


# -------------------------
# 给后续 random forest 用的文本特征
# 最推荐喂 proba + hit_count + zero_hit
# -------------------------
def build_nb_feature_frame(
    df_any: pd.DataFrame,
    food_model: TopKBernoulliNB,
    feeling_model: TopKBernoulliNB,
    soundtrack_model: TopKBernoulliNB
) -> pd.DataFrame:
    food_probs = food_model.predict_proba(df_any[FOOD_COL].fillna("missing"))
    feeling_probs = feeling_model.predict_proba(df_any[FEEL_COL].fillna("missing"))
    soundtrack_probs = soundtrack_model.predict_proba(df_any[SOUND_COL].fillna("missing"))

    food_hits = food_model.hit_count(df_any[FOOD_COL].fillna("missing"))
    feeling_hits = feeling_model.hit_count(df_any[FEEL_COL].fillna("missing"))
    soundtrack_hits = soundtrack_model.hit_count(df_any[SOUND_COL].fillna("missing"))

    out = pd.DataFrame({
        "food_nb_p0": food_probs[:, 0],
        "food_nb_p1": food_probs[:, 1],
        "food_nb_p2": food_probs[:, 2],
        "food_nb_pred": np.argmax(food_probs, axis=1),
        "food_nb_hit_count": food_hits,
        "food_nb_zero_hit": (food_hits == 0).astype(int),

        "feeling_nb_p0": feeling_probs[:, 0],
        "feeling_nb_p1": feeling_probs[:, 1],
        "feeling_nb_p2": feeling_probs[:, 2],
        "feeling_nb_pred": np.argmax(feeling_probs, axis=1),
        "feeling_nb_hit_count": feeling_hits,
        "feeling_nb_zero_hit": (feeling_hits == 0).astype(int),

        "soundtrack_nb_p0": soundtrack_probs[:, 0],
        "soundtrack_nb_p1": soundtrack_probs[:, 1],
        "soundtrack_nb_p2": soundtrack_probs[:, 2],
        "soundtrack_nb_pred": np.argmax(soundtrack_probs, axis=1),
        "soundtrack_nb_hit_count": soundtrack_hits,
        "soundtrack_nb_zero_hit": (soundtrack_hits == 0).astype(int),
    }, index=df_any.index)

    return out


# -------------------------
# 一个最小可用示例
# 真正使用时，把 df_train 换成你外层 split 的 85% training set
# -------------------------
if __name__ == "__main__":
    df = load_cleaned_data(CSV_PATH)

    # 这里只是示例，真正实验时请换成你自己的 outer split train set
    df_train = df.sample(frac=0.85, random_state=42)
    df_val = df.drop(df_train.index)

    food_model, feeling_model, soundtrack_model = train_text_nb_models(
        df_train,
        food_k=FOOD_K,
        feeling_k=FEELING_K,
        soundtrack_k=SOUNDTRACK_K,
        alpha=NB_ALPHA
    )

    # 生成给后续 random forest 用的特征
    train_nb_features = build_nb_feature_frame(df_train, food_model, feeling_model, soundtrack_model)
    val_nb_features = build_nb_feature_frame(df_val, food_model, feeling_model, soundtrack_model)

    print(train_nb_features.head())
    print(val_nb_features.head())

    # 看每个模型当前训练出来的 top-k 词
    print("Food vocab:", food_model.vocab_)
    print("Feeling vocab:", feeling_model.vocab_)
    print("Soundtrack vocab:", soundtrack_model.vocab_)



      food_nb_p0  food_nb_p1  food_nb_p2  food_nb_pred  food_nb_hit_count  \
29      0.810753    0.090269    0.098979             0                  1   
99      0.807661    0.084283    0.108057             0                  1   
678     0.336988    0.333333    0.329678             0                  0   
1399    0.700229    0.135707    0.164064             0                  1   
185     0.700229    0.135707    0.164064             0                  1   

      food_nb_zero_hit  feeling_nb_p0  feeling_nb_p1  feeling_nb_p2  \
29                   0       0.336988       0.333333       0.329678   
99                   0       0.387666       0.336652       0.275682   
678                  1       0.958097       0.036580       0.005323   
1399                 0       0.003464       0.143902       0.852634   
185                  0       0.999465       0.000416       0.000119   

      feeling_nb_pred  feeling_nb_hit_count  feeling_nb_zero_hit  \
29                  0                     

In [31]:
for i in range(1000):
    result = run_rf_train_val(
        "cleaned_data_final.csv",
        test_size=0.16,
        random_state=i
    )
    print(
        f"random_state={i}, "
        f"train_acc={result['train_acc']:.4f}, "
        f"val_acc={result['val_acc']:.4f}"
    )

train size: 1352
validation size: 258
training accuracy: 1.0
validation accuracy: 0.876
random_state=0, train_acc=1.0000, val_acc=0.8760
train size: 1352
validation size: 258
training accuracy: 1.0
validation accuracy: 0.9109
random_state=1, train_acc=1.0000, val_acc=0.9109
train size: 1352
validation size: 258
training accuracy: 1.0
validation accuracy: 0.8992
random_state=2, train_acc=1.0000, val_acc=0.8992
train size: 1352
validation size: 258
training accuracy: 1.0
validation accuracy: 0.8915
random_state=3, train_acc=1.0000, val_acc=0.8915
train size: 1352
validation size: 258
training accuracy: 1.0
validation accuracy: 0.9109
random_state=4, train_acc=1.0000, val_acc=0.9109
train size: 1352
validation size: 258
training accuracy: 1.0
validation accuracy: 0.907
random_state=5, train_acc=1.0000, val_acc=0.9070
train size: 1352
validation size: 258
training accuracy: 1.0
validation accuracy: 0.9186
random_state=6, train_acc=1.0000, val_acc=0.9186
train size: 1352
validation size: 25

KeyboardInterrupt: 

In [9]:
import numpy as np
import pandas as pd
from itertools import product
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


def get_base_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude_cols = {LABEL_COL, FOOD_COL, FEEL_COL, SOUND_COL}
    candidate_cols = [col for col in df.columns if col not in exclude_cols]

    # 只保留数值/布尔列，避免把字符串列送进 RandomForest
    numeric_cols = []
    for col in candidate_cols:
        if pd.api.types.is_numeric_dtype(df[col]) or pd.api.types.is_bool_dtype(df[col]):
            numeric_cols.append(col)

    return numeric_cols


def build_rf_feature_matrix(
    df_any: pd.DataFrame,
    food_model: TopKBernoulliNB,
    feeling_model: TopKBernoulliNB,
    soundtrack_model: TopKBernoulliNB
) -> pd.DataFrame:
    base_cols = get_base_feature_cols(df_any)
    base_X = df_any[base_cols].copy()

    nb_X = build_nb_feature_frame(
        df_any,
        food_model,
        feeling_model,
        soundtrack_model
    )

    X = pd.concat(
        [base_X.reset_index(drop=True), nb_X.reset_index(drop=True)],
        axis=1
    )
    return X


def train_rf_with_nb_features(
    df_train: pd.DataFrame,
    food_k: int,
    feeling_k: int,
    soundtrack_k: int,
    nb_alpha: float,
    n_estimators: int,
    max_depth,
    min_samples_leaf: int,
    max_features,
    bootstrap: bool,
    random_state: int
):
    food_model, feeling_model, soundtrack_model = train_text_nb_models(
        df_train,
        food_k=food_k,
        feeling_k=feeling_k,
        soundtrack_k=soundtrack_k,
        alpha=nb_alpha
    )

    X_train = build_rf_feature_matrix(
        df_train,
        food_model,
        feeling_model,
        soundtrack_model
    )
    y_train = df_train[LABEL_COL].to_numpy()

    rf_model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        random_state=random_state,
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)

    return rf_model, food_model, feeling_model, soundtrack_model


def predict_rf_from_models(
    df_any: pd.DataFrame,
    rf_model: RandomForestClassifier,
    food_model: TopKBernoulliNB,
    feeling_model: TopKBernoulliNB,
    soundtrack_model: TopKBernoulliNB
) -> pd.DataFrame:
    X_any = build_rf_feature_matrix(
        df_any,
        food_model,
        feeling_model,
        soundtrack_model
    )

    pred = rf_model.predict(X_any)

    out = df_any.copy()
    out["pred"] = pred
    return out


def evaluate_one_param_combo_kfold(
    df: pd.DataFrame,
    food_k: int,
    feeling_k: int,
    soundtrack_k: int,
    nb_alpha: float,
    n_estimators: int,
    max_depth,
    min_samples_leaf: int,
    max_features,
    bootstrap: bool,
    n_splits: int = 5,
    random_state: int = 42
) -> dict:
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    fold_scores = []
    X_dummy = df.drop(columns=[LABEL_COL])
    y = df[LABEL_COL].to_numpy()

    for train_idx, val_idx in skf.split(X_dummy, y):
        df_train = df.iloc[train_idx].copy()
        df_val = df.iloc[val_idx].copy()

        rf_model, food_model, feeling_model, soundtrack_model = train_rf_with_nb_features(
            df_train=df_train,
            food_k=food_k,
            feeling_k=feeling_k,
            soundtrack_k=soundtrack_k,
            nb_alpha=nb_alpha,
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            bootstrap=bootstrap,
            random_state=random_state
        )

        val_pred_df = predict_rf_from_models(
            df_val,
            rf_model,
            food_model,
            feeling_model,
            soundtrack_model
        )

        val_acc = accuracy_score(df_val[LABEL_COL], val_pred_df["pred"])
        fold_scores.append(val_acc)

    return {
        "food_k": food_k,
        "feeling_k": feeling_k,
        "soundtrack_k": soundtrack_k,
        "nb_alpha": nb_alpha,
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "max_features": max_features,
        "bootstrap": bootstrap,
        "fold_scores": fold_scores,
        "mean_val_acc": float(np.mean(fold_scores)),
        "std_val_acc": float(np.std(fold_scores)),
    }


def tune_rf_nb_hyperparameters_kfold(
    csv_path: str = CSV_PATH,
    n_splits: int = 5,
    random_state: int = 42,
    food_k_grid=None,
    feeling_k_grid=None,
    soundtrack_k_grid=None,
    nb_alpha_grid=None,
    n_estimators_grid=None,
    max_depth_grid=None,
    min_samples_leaf_grid=None,
    max_features_grid=None,
    bootstrap_grid=None
):
    df = load_cleaned_data(csv_path)

    if food_k_grid is None:
        food_k_grid = [20, 30, 40, 50]
    if feeling_k_grid is None:
        feeling_k_grid = [10, 20, 30, 40]
    if soundtrack_k_grid is None:
        soundtrack_k_grid = [20, 30, 40, 50]
    if nb_alpha_grid is None:
        nb_alpha_grid = [1.0]
    if n_estimators_grid is None:
        n_estimators_grid = [200, 300, 500]
    if max_depth_grid is None:
        max_depth_grid = [None, 10, 20, 30]
    if min_samples_leaf_grid is None:
        min_samples_leaf_grid = [1, 2, 4]
    if max_features_grid is None:
        max_features_grid = ["sqrt", "log2", None]
    if bootstrap_grid is None:
        bootstrap_grid = [True, False]

    param_grid = list(product(
        food_k_grid,
        feeling_k_grid,
        soundtrack_k_grid,
        nb_alpha_grid,
        n_estimators_grid,
        max_depth_grid,
        min_samples_leaf_grid,
        max_features_grid,
        bootstrap_grid
    ))

    all_results = []

    for combo_id, params in enumerate(param_grid, start=1):
        (
            food_k,
            feeling_k,
            soundtrack_k,
            nb_alpha,
            n_estimators,
            max_depth,
            min_samples_leaf,
            max_features,
            bootstrap
        ) = params

        result = evaluate_one_param_combo_kfold(
            df=df,
            food_k=food_k,
            feeling_k=feeling_k,
            soundtrack_k=soundtrack_k,
            nb_alpha=nb_alpha,
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            bootstrap=bootstrap,
            n_splits=n_splits,
            random_state=random_state
        )

        all_results.append(result)

        print(
            f"[{combo_id}/{len(param_grid)}] "
            f"food_k={food_k}, feeling_k={feeling_k}, soundtrack_k={soundtrack_k}, "
            f"n_estimators={n_estimators}, max_depth={max_depth}, "
            f"min_samples_leaf={min_samples_leaf}, max_features={max_features}, "
            f"bootstrap={bootstrap} "
            f"=> mean_val_acc={result['mean_val_acc']:.4f}, std={result['std_val_acc']:.4f}"
        )

    results_df = pd.DataFrame(all_results)
    results_df = results_df.sort_values(
        by=["mean_val_acc", "std_val_acc"],
        ascending=[False, True]
    ).reset_index(drop=True)

    best_row = results_df.iloc[0]

    print("\n=========================")
    print("Best Hyperparameter Combination")
    print("=========================")
    print("food_k:", best_row["food_k"])
    print("feeling_k:", best_row["feeling_k"])
    print("soundtrack_k:", best_row["soundtrack_k"])
    print("nb_alpha:", best_row["nb_alpha"])
    print("n_estimators:", best_row["n_estimators"])
    print("max_depth:", best_row["max_depth"])
    print("min_samples_leaf:", best_row["min_samples_leaf"])
    print("max_features:", best_row["max_features"])
    print("bootstrap:", best_row["bootstrap"])
    print("best mean validation accuracy:", round(best_row["mean_val_acc"], 4))
    print("best std:", round(best_row["std_val_acc"], 4))
    print("fold scores:", best_row["fold_scores"])

    return {
        "best_params": {
            "food_k": best_row["food_k"],
            "feeling_k": best_row["feeling_k"],
            "soundtrack_k": best_row["soundtrack_k"],
            "nb_alpha": best_row["nb_alpha"],
            "n_estimators": best_row["n_estimators"],
            "max_depth": best_row["max_depth"],
            "min_samples_leaf": best_row["min_samples_leaf"],
            "max_features": best_row["max_features"],
            "bootstrap": best_row["bootstrap"],
        },
        "best_mean_val_acc": best_row["mean_val_acc"],
        "best_std_val_acc": best_row["std_val_acc"],
        "results_df": results_df
    }


small_food_k_grid = [30, 50]
small_feeling_k_grid = [20, 40]
small_soundtrack_k_grid = [30, 50]

small_nb_alpha_grid = [1.0]

small_n_estimators_grid = [200, 300]
small_max_depth_grid = [None, 20]
small_min_samples_leaf_grid = [1, 2]
small_max_features_grid = ["sqrt", None]
small_bootstrap_grid = [True]

search_result = tune_rf_nb_hyperparameters_kfold(
    csv_path="cleaned_data_final.csv",
    n_splits=5,
    random_state=42,
    food_k_grid=small_food_k_grid,
    feeling_k_grid=small_feeling_k_grid,
    soundtrack_k_grid=small_soundtrack_k_grid,
    nb_alpha_grid=small_nb_alpha_grid,
    n_estimators_grid=small_n_estimators_grid,
    max_depth_grid=small_max_depth_grid,
    min_samples_leaf_grid=small_min_samples_leaf_grid,
    max_features_grid=small_max_features_grid,
    bootstrap_grid=small_bootstrap_grid
)

results_df = search_result["results_df"]

print(results_df.head(10)[[
    "food_k",
    "feeling_k",
    "soundtrack_k",
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "max_features",
    "bootstrap",
    "mean_val_acc",
    "std_val_acc"
]])

results_df.to_csv("small_grid_results.csv", index=False)

[1/128] food_k=30, feeling_k=20, soundtrack_k=30, n_estimators=200, max_depth=None, min_samples_leaf=1, max_features=sqrt, bootstrap=True => mean_val_acc=0.8540, std=0.0119
[2/128] food_k=30, feeling_k=20, soundtrack_k=30, n_estimators=200, max_depth=None, min_samples_leaf=1, max_features=None, bootstrap=True => mean_val_acc=0.8528, std=0.0169


KeyboardInterrupt: 

In [10]:

import time
import numpy as np
import pandas as pd
from itertools import product
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


def get_base_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude_cols = {LABEL_COL, FOOD_COL, FEEL_COL, SOUND_COL}
    candidate_cols = [col for col in df.columns if col not in exclude_cols]

    numeric_cols = []
    for col in candidate_cols:
        if pd.api.types.is_numeric_dtype(df[col]) or pd.api.types.is_bool_dtype(df[col]):
            numeric_cols.append(col)

    return numeric_cols


def build_rf_feature_matrix(
    df_any: pd.DataFrame,
    food_model: TopKBernoulliNB,
    feeling_model: TopKBernoulliNB,
    soundtrack_model: TopKBernoulliNB
) -> pd.DataFrame:
    base_cols = get_base_feature_cols(df_any)
    base_X = df_any[base_cols].copy()

    nb_X = build_nb_feature_frame(
        df_any,
        food_model,
        feeling_model,
        soundtrack_model
    )

    X = pd.concat(
        [base_X.reset_index(drop=True), nb_X.reset_index(drop=True)],
        axis=1
    )
    return X


def train_rf_with_nb_features(
    df_train: pd.DataFrame,
    food_k: int,
    feeling_k: int,
    soundtrack_k: int,
    nb_alpha: float,
    n_estimators: int,
    max_depth,
    min_samples_leaf: int,
    max_features,
    bootstrap: bool,
    random_state: int
):
    food_model, feeling_model, soundtrack_model = train_text_nb_models(
        df_train,
        food_k=food_k,
        feeling_k=feeling_k,
        soundtrack_k=soundtrack_k,
        alpha=nb_alpha
    )

    X_train = build_rf_feature_matrix(
        df_train,
        food_model,
        feeling_model,
        soundtrack_model
    )
    y_train = df_train[LABEL_COL].to_numpy()

    rf_model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        random_state=random_state,
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)

    return rf_model, food_model, feeling_model, soundtrack_model


def predict_rf_from_models(
    df_any: pd.DataFrame,
    rf_model: RandomForestClassifier,
    food_model: TopKBernoulliNB,
    feeling_model: TopKBernoulliNB,
    soundtrack_model: TopKBernoulliNB
) -> pd.DataFrame:
    X_any = build_rf_feature_matrix(
        df_any,
        food_model,
        feeling_model,
        soundtrack_model
    )

    pred = rf_model.predict(X_any)

    out = df_any.copy()
    out["pred"] = pred
    return out


def evaluate_one_param_combo_kfold(
    df: pd.DataFrame,
    food_k: int,
    feeling_k: int,
    soundtrack_k: int,
    nb_alpha: float,
    n_estimators: int,
    max_depth,
    min_samples_leaf: int,
    max_features,
    bootstrap: bool,
    n_splits: int = 3,
    random_state: int = 42
) -> dict:
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    fold_scores = []
    X_dummy = df.drop(columns=[LABEL_COL])
    y = df[LABEL_COL].to_numpy()

    for fold_id, (train_idx, val_idx) in enumerate(skf.split(X_dummy, y), start=1):
        df_train = df.iloc[train_idx].copy()
        df_val = df.iloc[val_idx].copy()

        rf_model, food_model, feeling_model, soundtrack_model = train_rf_with_nb_features(
            df_train=df_train,
            food_k=food_k,
            feeling_k=feeling_k,
            soundtrack_k=soundtrack_k,
            nb_alpha=nb_alpha,
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            bootstrap=bootstrap,
            random_state=random_state
        )

        val_pred_df = predict_rf_from_models(
            df_val,
            rf_model,
            food_model,
            feeling_model,
            soundtrack_model
        )

        val_acc = accuracy_score(df_val[LABEL_COL], val_pred_df["pred"])
        fold_scores.append(val_acc)

    return {
        "food_k": food_k,
        "feeling_k": feeling_k,
        "soundtrack_k": soundtrack_k,
        "nb_alpha": nb_alpha,
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "max_features": max_features,
        "bootstrap": bootstrap,
        "fold_scores": fold_scores,
        "mean_val_acc": float(np.mean(fold_scores)),
        "std_val_acc": float(np.std(fold_scores)),
    }


def tune_rf_nb_hyperparameters_kfold(
    csv_path: str = CSV_PATH,
    n_splits: int = 3,
    random_state: int = 42,
    food_k_grid=None,
    feeling_k_grid=None,
    soundtrack_k_grid=None,
    nb_alpha_grid=None,
    n_estimators_grid=None,
    max_depth_grid=None,
    min_samples_leaf_grid=None,
    max_features_grid=None,
    bootstrap_grid=None
):
    df = load_cleaned_data(csv_path)

    if food_k_grid is None:
        food_k_grid = [30, 40, 50, 60]
    if feeling_k_grid is None:
        feeling_k_grid = [35, 40, 45, 50]
    if soundtrack_k_grid is None:
        soundtrack_k_grid = [40, 50, 60]
    if nb_alpha_grid is None:
        nb_alpha_grid = [1.0]
    if n_estimators_grid is None:
        n_estimators_grid = [300, 500]
    if max_depth_grid is None:
        max_depth_grid = [None, 20, 30]
    if min_samples_leaf_grid is None:
        min_samples_leaf_grid = [1, 2]
    if max_features_grid is None:
        max_features_grid = ["sqrt", "log2"]
    if bootstrap_grid is None:
        bootstrap_grid = [True, False]

    param_grid = list(product(
        food_k_grid,
        feeling_k_grid,
        soundtrack_k_grid,
        nb_alpha_grid,
        n_estimators_grid,
        max_depth_grid,
        min_samples_leaf_grid,
        max_features_grid,
        bootstrap_grid
    ))

    print("total combinations:", len(param_grid))
    print("total model fits:", len(param_grid) * n_splits)

    all_results = []
    start_time = time.time()

    for combo_id, params in enumerate(param_grid, start=1):
        (
            food_k,
            feeling_k,
            soundtrack_k,
            nb_alpha,
            n_estimators,
            max_depth,
            min_samples_leaf,
            max_features,
            bootstrap
        ) = params

        result = evaluate_one_param_combo_kfold(
            df=df,
            food_k=food_k,
            feeling_k=feeling_k,
            soundtrack_k=soundtrack_k,
            nb_alpha=nb_alpha,
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            bootstrap=bootstrap,
            n_splits=n_splits,
            random_state=random_state
        )

        all_results.append(result)

        elapsed = time.time() - start_time
        avg_per_combo = elapsed / combo_id
        remain = avg_per_combo * (len(param_grid) - combo_id)

        print(
            f"[{combo_id}/{len(param_grid)}] "
            f"food_k={food_k}, feeling_k={feeling_k}, soundtrack_k={soundtrack_k}, "
            f"n_estimators={n_estimators}, max_depth={max_depth}, "
            f"min_samples_leaf={min_samples_leaf}, max_features={max_features}, "
            f"bootstrap={bootstrap} "
            f"=> mean_val_acc={result['mean_val_acc']:.4f}, std={result['std_val_acc']:.4f} "
            f"| elapsed={elapsed/60:.1f} min, remain≈{remain/60:.1f} min"
        )

    results_df = pd.DataFrame(all_results)
    results_df = results_df.sort_values(
        by=["mean_val_acc", "std_val_acc"],
        ascending=[False, True]
    ).reset_index(drop=True)

    best_row = results_df.iloc[0]

    print("\n=========================")
    print("Best Hyperparameter Combination")
    print("=========================")
    print("food_k:", best_row["food_k"])
    print("feeling_k:", best_row["feeling_k"])
    print("soundtrack_k:", best_row["soundtrack_k"])
    print("nb_alpha:", best_row["nb_alpha"])
    print("n_estimators:", best_row["n_estimators"])
    print("max_depth:", best_row["max_depth"])
    print("min_samples_leaf:", best_row["min_samples_leaf"])
    print("max_features:", best_row["max_features"])
    print("bootstrap:", best_row["bootstrap"])
    print("best mean validation accuracy:", round(best_row["mean_val_acc"], 4))
    print("best std:", round(best_row["std_val_acc"], 4))
    print("fold scores:", best_row["fold_scores"])

    return {
        "best_params": {
            "food_k": best_row["food_k"],
            "feeling_k": best_row["feeling_k"],
            "soundtrack_k": best_row["soundtrack_k"],
            "nb_alpha": best_row["nb_alpha"],
            "n_estimators": best_row["n_estimators"],
            "max_depth": best_row["max_depth"],
            "min_samples_leaf": best_row["min_samples_leaf"],
            "max_features": best_row["max_features"],
            "bootstrap": best_row["bootstrap"],
        },
        "best_mean_val_acc": best_row["mean_val_acc"],
        "best_std_val_acc": best_row["std_val_acc"],
        "results_df": results_df
    }


search_result = tune_rf_nb_hyperparameters_kfold(
    csv_path="cleaned_data_final.csv",
    n_splits=3,
    random_state=42,
    food_k_grid=[30, 40, 50, 60],
    feeling_k_grid=[35, 40, 45, 50],
    soundtrack_k_grid=[40, 50, 60],
    nb_alpha_grid=[1.0],
    n_estimators_grid=[300, 500],
    max_depth_grid=[None, 20, 30],
    min_samples_leaf_grid=[1, 2],
    max_features_grid=["sqrt", "log2"],
    bootstrap_grid=[True, False]
)

results_df = search_result["results_df"]

print(results_df.head(15)[[
    "food_k",
    "feeling_k",
    "soundtrack_k",
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "max_features",
    "bootstrap",
    "mean_val_acc",
    "std_val_acc"
]])

results_df.to_csv("one_hour_grid_results.csv", index=False)

total combinations: 2304
total model fits: 6912
[1/2304] food_k=30, feeling_k=35, soundtrack_k=40, n_estimators=300, max_depth=None, min_samples_leaf=1, max_features=sqrt, bootstrap=True => mean_val_acc=0.8634, std=0.0083 | elapsed=0.0 min, remain≈47.8 min
[2/2304] food_k=30, feeling_k=35, soundtrack_k=40, n_estimators=300, max_depth=None, min_samples_leaf=1, max_features=sqrt, bootstrap=False => mean_val_acc=0.8615, std=0.0077 | elapsed=0.0 min, remain≈44.1 min
[3/2304] food_k=30, feeling_k=35, soundtrack_k=40, n_estimators=300, max_depth=None, min_samples_leaf=1, max_features=log2, bootstrap=True => mean_val_acc=0.8627, std=0.0112 | elapsed=0.1 min, remain≈45.0 min
[4/2304] food_k=30, feeling_k=35, soundtrack_k=40, n_estimators=300, max_depth=None, min_samples_leaf=1, max_features=log2, bootstrap=False => mean_val_acc=0.8596, std=0.0077 | elapsed=0.1 min, remain≈43.7 min


KeyboardInterrupt: 

In [ ]:
search_result = tune_rf_nb_hyperparameters_kfold(
    csv_path="cleaned_data_final.csv",
    n_splits=3,
    random_state=42,
    food_k_grid=[24, 28, 32, 36, 40],
    feeling_k_grid=[38, 40, 42, 44, 46],
    soundtrack_k_grid=[45, 50, 55, 60, 65],
    nb_alpha_grid=[1.0],
    n_estimators_grid=[500, 700],
    max_depth_grid=[None, 20],
    min_samples_leaf_grid=[1, 2],
    max_features_grid=["sqrt", "log2"],
    bootstrap_grid=[True]
)

results_df = search_result["results_df"]

print(results_df.head(20)[[
    "food_k",
    "feeling_k",
    "soundtrack_k",
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "max_features",
    "bootstrap",
    "mean_val_acc",
    "std_val_acc"
]])

results_df.to_csv("refined_90min_grid_results.csv", index=False)

total combinations: 2000
total model fits: 6000
[1/2000] food_k=24, feeling_k=38, soundtrack_k=45, n_estimators=500, max_depth=None, min_samples_leaf=1, max_features=sqrt, bootstrap=True => mean_val_acc=0.8627, std=0.0097 | elapsed=0.0 min, remain≈70.6 min
[2/2000] food_k=24, feeling_k=38, soundtrack_k=45, n_estimators=500, max_depth=None, min_samples_leaf=1, max_features=log2, bootstrap=True => mean_val_acc=0.8634, std=0.0101 | elapsed=0.1 min, remain≈63.6 min
[3/2000] food_k=24, feeling_k=38, soundtrack_k=45, n_estimators=500, max_depth=None, min_samples_leaf=2, max_features=sqrt, bootstrap=True => mean_val_acc=0.8640, std=0.0109 | elapsed=0.1 min, remain≈61.4 min
[4/2000] food_k=24, feeling_k=38, soundtrack_k=45, n_estimators=500, max_depth=None, min_samples_leaf=2, max_features=log2, bootstrap=True => mean_val_acc=0.8652, std=0.0114 | elapsed=0.1 min, remain≈60.4 min
[5/2000] food_k=24, feeling_k=38, soundtrack_k=45, n_estimators=500, max_depth=20, min_samples_leaf=1, max_features=